# Customer Churn Forecasting - Decision Tree
## Module 4: Non-Linear Tree Classification

Decision Tree classifier with controlled depth to prevent overfitting, providing interpretable decision rules for churn prediction.

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_auc_score, roc_curve,
                             precision_score, recall_score, f1_score)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded successfully.')

## 4.1 Load and Prepare Data

In [ ]:
# Load and prepare the dataset (same preprocessing as previous notebooks)
if os.path.exists('WA_Fn-UseC_-Telco-Customer-Churn.csv'):
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
elif os.path.exists('telco_customer_churn.csv'):
    df = pd.read_csv('telco_customer_churn.csv')
else:
    raise FileNotFoundError('Dataset not found.')

# Preprocessing pipeline
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['tenure'] * df['MonthlyCharges'], inplace=True)
df.drop(columns=['customerID'], inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Feature engineering
df['AvgMonthlySpend'] = np.where(df['tenure'] > 0, df['TotalCharges'] / df['tenure'], df['MonthlyCharges'])
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['TotalServices'] = sum((df[col].isin(['Yes', 'Fiber optic', 'DSL'])).astype(int) for col in service_cols if col in df.columns)
df['ChargesPerService'] = np.where(df['TotalServices'] > 0, df['MonthlyCharges'] / df['TotalServices'], df['MonthlyCharges'])
contract_risk = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
df['ContractRisk'] = df['Contract'].map(contract_risk)

# Encode
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
binary_cols = [col for col in cat_cols if df[col].nunique() == 2]
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])
multi_cat_cols = [col for col in cat_cols if df[col].nunique() > 2]
df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True, dtype=int)

# Split
X = df.drop(columns=['Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f'Dataset prepared. Shape: {df.shape}')
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

## 4.2 Train Decision Tree Classifier

In [ ]:
# Train Decision Tree with constrained depth
dt_model = DecisionTreeClassifier(
    max_depth=6,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42
)
dt_model.fit(X_train, y_train)

# Predictions
y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1]

print('=' * 60)
print('DECISION TREE - RESULTS')
print('=' * 60)
print(f'\nAccuracy:  {accuracy_score(y_test, y_pred_dt)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred_dt)*100:.2f}%')
print(f'Recall:    {recall_score(y_test, y_pred_dt)*100:.2f}%')
print(f'F1-Score:  {f1_score(y_test, y_pred_dt):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_dt):.4f}')

print('\n' + '=' * 60)
print('CLASSIFICATION REPORT')
print('=' * 60)
print(classification_report(y_test, y_pred_dt, target_names=['Retained', 'Churned']))

## 4.3 Confusion Matrix

In [ ]:
# Confusion Matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'],
            annot_kws={'size': 16}, linewidths=1, linecolor='white')
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
ax.set_title('Decision Tree - Confusion Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('dt_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm_dt.ravel()
print(f'True Negatives:  {tn:,} | False Positives: {fp:,}')
print(f'False Negatives: {fn:,} | True Positives:  {tp:,}')

## 4.4 Decision Tree Visualization

In [ ]:
# Visualize the decision tree structure
plt.figure(figsize=(25, 12))
plot_tree(dt_model, 
          feature_names=X.columns.tolist(),
          class_names=['Retained', 'Churned'],
          filled=True, 
          rounded=True,
          fontsize=8,
          max_depth=3,  # Show first 3 levels for readability
          proportion=True)
plt.title('Decision Tree Structure (First 3 Levels)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('dt_tree_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Decision tree visualization saved.')

## 4.5 Feature Importance

In [ ]:
# Feature importance from Decision Tree
dt_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('Top 15 Most Important Features (Decision Tree):')
display(dt_importance.head(15))

# Visualize
top_15 = dt_importance.head(15)
plt.figure(figsize=(10, 7))
plt.barh(range(len(top_15)), top_15['Importance'].values, 
         color='#e67e22', edgecolor='black', alpha=0.85)
plt.yticks(range(len(top_15)), top_15['Feature'].values)
plt.xlabel('Feature Importance (Gini)', fontsize=12)
plt.title('Top 15 Feature Importances - Decision Tree', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dt_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cross-validation
cv_scores_dt = cross_val_score(dt_model, X_train, y_train, cv=5, scoring='roc_auc')
print('5-Fold Cross-Validation (ROC-AUC):')
for i, score in enumerate(cv_scores_dt, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'\n  Mean AUC: {cv_scores_dt.mean():.4f} (+/- {cv_scores_dt.std():.4f})')